In [56]:
# Initialize Otter
import otter
grader = otter.Notebook("hw7.ipynb")

# CPSC 330 - Applied Machine Learning 

## Homework 7: Word embeddings and topic modeling 

**Due date: See [deliverable due dates](https://ubc-cs.github.io/cpsc330-2025W2/#deliverable-due-dates-tentative)**.

## Imports

In [57]:
import os

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.decomposition import LatentDirichletAllocation

In [58]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

<br><br>

<!-- BEGIN QUESTION -->

<!-- BEGIN QUESTION -->

<div class="alert alert-info">
    
## Instructions
rubric={points}

You will earn points for following these instructions and successfully submitting your work on Gradescope.  

### Group wotk instructions

**You may work with a partner on this homework and submit your assignment as a group.** Below are some instructions on working as a group.  
- The maximum group size is 2.
  
- Use group work as an opportunity to collaborate and learn new things from each other. 
- Be respectful to each other and make sure you understand all the concepts in the assignment well. 
- It's your responsibility to make sure that the assignment is submitted by one of the group members before the deadline. 
- You can find the instructions on how to do group submission on Gradescope [here](https://help.gradescope.com/article/m5qz2xsnjy-student-add-group-members).
- If you would like to use late tokens for the homework, all group members must have the necessary late tokens available. Please note that the late tokens will be counted for all members of the group.   


### General submission instructions

- Please **read carefully
[Use of Generative AI policy](https://ubc-cs.github.io/cpsc330-2025W2/syllabus#use-of-generative-ai-in-the-course)** before starting the homework assignment. 
- **Run all cells before submitting:** Go to `Kernel -> Restart Kernel and Clear All Outputs`, then select `Run -> Run All Cells`. This ensures your notebook runs cleanly from start to finish without errors.
  
- **Submit your files on Gradescope.**  
   - Upload only your `.ipynb` file **with outputs displayed** and any required output files.
     
   - Do **not** submit other files from your repository.  
   - If you need help, see the [Gradescope Student Guide](https://lthub.ubc.ca/guides/gradescope-student-guide/).  
- **Check that outputs render properly.**  
   - Make sure all plots and outputs appear in your submission.
     
   - If your `.ipynb` file is too large and doesn't render on Gradescope, also upload a PDF or HTML version so the TAs can view your work.  
- **Keep execution order clean.**  
   - Execution numbers must start at "1" and increase in order.
     
   - Notebooks without visible outputs may not be graded.  
   - Out-of-order or missing execution numbers may result in mark deductions.  
- **Follow course submission guidelines:** Review the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions) for detailed guidance on completing and submitting assignments. 
   
</div>


_Points:_ 2

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 1:  Exploring pre-trained word embeddings <a name="1"></a>
<hr>

In lecture 18, we talked about natural language processing (NLP). Using pre-trained word embeddings is very common in NLP. It has been shown that pre-trained word embeddings work well on a variety of text classification tasks. These embeddings are created by training a model like Word2Vec on a huge corpus of text such as a dump of Wikipedia or a dump of the web crawl. 

A number of pre-trained word embeddings are available out there. Some popular ones are: 

- [GloVe](https://nlp.stanford.edu/projects/glove/)
    * trained using [the GloVe algorithm](https://nlp.stanford.edu/pubs/glove.pdf) 
    * published by Stanford University 
- [fastText pre-trained embeddings for 294 languages](https://fasttext.cc/docs/en/pretrained-vectors.html) 
    * trained using the fastText algorithm
    * published by Facebook
    
In this exercise, you will be exploring GloVe Wikipedia pre-trained embeddings. The code below loads the word vectors trained on Wikipedia using an algorithm called Glove. You'll need `gensim` package in your cpsc330 conda environment to run the code below. 

```
> conda activate cpsc330
> conda install -c anaconda gensim
```

In [59]:
import gensim
import gensim.downloader

print(list(gensim.downloader.info()["models"].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [60]:
# This will take a while to run when you run it for the first time.
import gensim.downloader as api

glove_wiki_vectors = api.load("glove-wiki-gigaword-100")

In [61]:
len(glove_wiki_vectors)

400000

There are 400,000 word vectors in this pre-trained model. 

Now that we have GloVe Wiki vectors loaded in `glove_wiki_vectors`, let's explore the embeddings. 

<br><br>

<!-- BEGIN QUESTION -->

### 1.1 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

- Come up with a list of 4 words of your choice and find similar words to these words using `glove_wiki_vectors` embeddings.

<div class="alert alert-warning">

Solution_1.1
    
</div>

_Points:_ 2

In [62]:
words = ["vicarious", "schism", "parabola", "pot"]

In [63]:
for w in words:
    print(f"\nClosest to '{w}':")
    for neighbor, score in glove_wiki_vectors.most_similar(w, topn=5):
        print(f"  {neighbor:15s}  {score:.3f}")


Closest to 'vicarious':
  thrill           0.557
  thrills          0.553
  voyeuristic      0.542
  pleasurable      0.521
  craving          0.518

Closest to 'schism':
  1054             0.713
  rift             0.664
  rifts            0.651
  schisms          0.582
  disunity         0.559

Closest to 'parabola':
  hyperbola        0.688
  ellipse          0.635
  tangent          0.585
  zigzag           0.564
  conic            0.551

Closest to 'pot':
  boiling          0.688
  soup             0.646
  chicken          0.632
  boil             0.615
  pots             0.614


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.2 Word similarity using pre-trained embeddings
rubric={points}

**Your tasks:**

1. Calculate cosine similarity for the following word pairs (`word_pairs`) using the [`similarity`](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) method of `glove_wiki_vectors`.

In [64]:
word_pairs = [
    ("coast", "shore"),
    ("clothes", "closet"),
    ("old", "new"),
    ("smart", "intelligent"),
    ("dog", "cat"),
    ("tree", "lawyer"),
]

<div class="alert alert-warning">

Solution_1.2
    
</div>

_Points:_ 2

In [65]:
similarities = []

for w1, w2 in word_pairs:
    score = glove_wiki_vectors.similarity(w1, w2)
    similarities.append((w1, w2, score))

In [66]:
sim_df = pd.DataFrame(similarities, columns=["word_1", "word_2", "cosine_similarity"])
sim_df

,word_1,word_2,cosine_similarity
0,coast,shore,0.700027
1,clothes,closet,0.546276
2,old,new,0.643249
3,smart,intelligent,0.755273
4,dog,cat,0.879808
5,tree,lawyer,0.076719


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.3 Representation of all words in English
rubric={points}

**Your tasks:**

1. The vocabulary size of Wikipedia embeddings is quite large. The `test_words` list below contains a few new words (called neologisms) and biomedical domain-specific abbreviations. Write code to check whether `glove_wiki_vectors` have representation for these words or not. 
> If a given word `word` is in the vocabulary, `word in glove_wiki_vectors` will return True. 

In [67]:
test_words = [
    "covididiot",
    "fomo",
    "frenemies",
    "anthropause",
    "photobomb",
    "selfie",
    "pxg",  # Abbreviation for pseudoexfoliative glaucoma
    "pacg",  # Abbreviation for primary angle closure glaucoma
    "cct",  # Abbreviation for central corneal thickness
    "escc",  # Abbreviation for esophageal squamous cell carcinoma
]

<div class="alert alert-warning">

Solution_1_3
    
</div>

_Points:_ 2

In [68]:
vocab_check = pd.DataFrame(
    {
        "word": test_words,
        "in_glove_vocab": [w in glove_wiki_vectors for w in test_words],
    }
)

vocab_check

,word,in_glove_vocab
0,covididiot,False
1,fomo,False
2,frenemies,True
3,anthropause,False
4,photobomb,False
5,selfie,False
6,pxg,False
7,pacg,False
8,cct,True
9,escc,True


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.4 Stereotypes and biases in embeddings
rubric={points}

Word vectors contain lots of useful information. But they also contain stereotypes and biases of the texts they were trained on. In the lecture, we saw an example of gender bias in Google News word embeddings. Here we are using pre-trained embeddings trained on Wikipedia data. 

**Your tasks:**

1. Explore whether there are any worrisome biases or stereotypes present in these embeddings by trying out at least 4 examples. You can use the following two methods or other methods of your choice to explore this. 
    - the `analogy` function below which gives word analogies (an example shown below)
    - [similarity](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=similarity#gensim.models.keyedvectors.KeyedVectors.similarity) or [distance](https://radimrehurek.com/gensim/models/keyedvectors.html?highlight=distance#gensim.models.keyedvectors.KeyedVectors.distances) methods (an example is shown below)

> Note that most of the recent embeddings are de-biased. But you might still observe some biases in them. Also, not all stereotypes present in pre-trained embeddings are necessarily bad. But you should be aware of them when you use them in your models. 

In [69]:
def analogy(word1, word2, word3, model=glove_wiki_vectors):
    """
    Returns analogy word using the given model.

    Parameters
    --------------
    word1 : (str)
        word1 in the analogy relation
    word2 : (str)
        word2 in the analogy relation
    word3 : (str)
        word3 in the analogy relation
    model :
        word embedding model

    Returns
    ---------------
        pd.dataframe
    """
    print("%s : %s :: %s : ?" % (word1, word2, word3))
    sim_words = model.most_similar(positive=[word3, word2], negative=[word1])
    return pd.DataFrame(sim_words, columns=["Analogy word", "Score"])

Examples of using analogy to explore biases and stereotypes.  

In [70]:
analogy("man", "doctor", "woman")

man : doctor :: woman : ?


,Analogy word,Score
0,nurse,0.773523
1,physician,0.718943
2,doctors,0.682433
3,patient,0.675068
4,dentist,0.672603
5,pregnant,0.664246
6,medical,0.652045
7,nursing,0.645348
8,mother,0.639333
9,hospital,0.638750


In [71]:
glove_wiki_vectors.similarity("aboriginal", "success")

np.float32(0.14283238)

In [72]:
glove_wiki_vectors.similarity("white", "success")

np.float32(0.351824)

<div class="alert alert-warning">

Solution_1_4
    
</div>

_Points:_ 4

In [73]:
analogy("man", "engineer", "woman")

man : engineer :: woman : ?


,Analogy word,Score
0,technician,0.662020
1,educator,0.611515
2,engineers,0.592235
3,surgeon,0.588020
4,contractor,0.587330
5,engineering,0.568600
6,nurse,0.556342
7,chemist,0.554719
8,pioneer,0.546008
9,physician,0.545960


In [74]:
analogy("man", "leader", "woman")

man : leader :: woman : ?


,Analogy word,Score
0,opposition,0.688884
1,party,0.685079
2,lawmaker,0.651261
3,candidate,0.645046
4,democratic,0.622170
5,activist,0.600771
6,president,0.599504
7,leadership,0.595807
8,politician,0.595188
9,leaders,0.589962


In [75]:
analogy("man", "scientist", "woman")

man : scientist :: woman : ?


,Analogy word,Score
0,researcher,0.777012
1,anthropologist,0.684751
2,sociologist,0.670103
3,psychologist,0.658697
4,physicist,0.657407
5,professor,0.650351
6,biologist,0.643093
7,geneticist,0.641517
8,biochemist,0.623625
9,expert,0.613421


In [76]:
analogy("rich", "white", "black")

rich : white :: black : ?


,Analogy word,Score
0,blue,0.686522
1,red,0.660400
2,gray,0.657767
3,pink,0.643225
4,brown,0.627526
5,yellow,0.625840
6,wearing,0.609402
7,green,0.604726
8,purple,0.594518
9,stripes,0.593542


In [77]:
pairs = [
    ("white", "success"),
    ("black", "success"),
    ("aboriginal", "success"),
    ("immigrant", "crime"),
    ("native", "crime"),
    ("muslim", "terrorism"),
    ("christian", "terrorism"),
    ("young", "technology"),
    ("old", "technology"),
]

rows = []
for a, b in pairs:
    try:
        rows.append((a, b, float(glove_wiki_vectors.similarity(a, b))))
    except KeyError:
        rows.append((a, b, None))

pd.DataFrame(rows, columns=["word_1", "word_2", "cosine_similarity"])

,word_1,word_2,cosine_similarity
0,white,success,0.351824
1,black,success,0.372831
2,aboriginal,success,0.142832
3,immigrant,crime,0.479125
4,native,crime,0.229824
5,muslim,terrorism,0.482661
6,christian,terrorism,0.267537
7,young,technology,0.374287
8,old,technology,0.326070


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 1.5 Discussion
rubric={points}

**Your tasks:**
1. Discuss your observations from 1.4. Are there any worrisome biases in these embeddings trained on Wikipedia?   
2. Give an example of how using embeddings with biases could cause harm in the real world.

<div class="alert alert-warning">

Solution_1_5
    
</div>

_Points:_ 4

### Biases Identified:

**1. Gender Bias in Professional Roles**
- `man : doctor :: woman : ?` returns "nurse" rather than "doctor"
- `man : engineer :: woman : ?` returns "technician" rather than "engineer"
- This suggests the embedding associates men and women differently with different professions.

**2. Racial/Ethnic Disparities in Success**
- `"white" + "success"` similarity: **0.351824**
- `"black" + "success"` similarity: **0.372831**
- `"aboriginal" + "success"` similarity: **0.142832**
- "aboriginal" has much weaker association with success compared to other groups.

**3. Religion-Violence Associations**
- `"muslim" + "terrorism"` similarity: **0.482661**
- `"christian" + "terrorism"` similarity: **0.267537**
- Muslims are disproportionately associated with terrorism.

**4. Immigrant-Crime Association**
- `"immigrant" + "crime"` similarity: **0.479125**
- `"native" + "crime"` similarity: **0.229824**
- This reflects a stereotype linking immigration with crime.

### Real-World Harm Example:

- A woman's resume for an engineering role might be ranked lower or categorized as a "technician" role by the ATS system during hiring
- Any vector RAG systems or other AI applications that use text embeddings could produce biased responses due to the bias in embeddings

There are some worrisome biases in the embeddings trained on Wikipedia. However, should we always be trying to eliminate these biases from embedding models? I'm not sure about that. But I do think we should be aware of these biases when we are using the emeddings in the real world.

<!-- END QUESTION -->

<br><br><br><br>

## Exercise 2: Topic modeling 

The goal of topic modeling is discovering high-level themes in a large collection of texts. 

In this homework, you will explore topics in [the 20 newsgroups text dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_20newsgroups.html) using `scikit-learn`'s `LatentDirichletAllocation` (LDA) model. 

Usually, topic modeling is used for discovering abstract "topics" that occur in a collection of documents when you do not know the actual topics present in the documents. But 20 newsgroups text dataset is labeled with categories (e.g., sports, hardware, religion), and you will be able to cross-check the topics discovered by your model with these available topics. 

The starter code below loads the train and test portion of the data and convert the train portion into a pandas DataFrame. For speed, we will only consider documents with the following 8 categories. 

In [78]:
from sklearn.datasets import fetch_20newsgroups

In [79]:
cats = [
    "rec.sport.hockey",
    "rec.sport.baseball",
    "soc.religion.christian",
    "alt.atheism",
    "comp.graphics",
    "comp.windows.x",
    "talk.politics.mideast",
    "talk.politics.guns",
]  # We'll only consider these categories out of 20 categories for speed.

newsgroups_train = fetch_20newsgroups(
    subset="train", remove=("headers", "footers", "quotes"), categories=cats
)
X_news_train, y_news_train = newsgroups_train.data, newsgroups_train.target
df = pd.DataFrame(X_news_train, columns=["text"])
df["target"] = y_news_train
df["target_name"] = [
    newsgroups_train.target_names[target] for target in newsgroups_train.target
]
df

text  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [80]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.windows.x',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast']

<br><br>

<!-- BEGIN QUESTION -->

### 2.1 Preprocessing using [spaCy](https://spacy.io/)
rubric={points}

Preprocessing is a crucial step before carrying out topic modeling and it markedly affects topic modeling results. In this exercise, you'll prepare the data using [spaCy](https://spacy.io/) for topic modeling. 

**Your tasks:** 

- Write code using [spaCy](https://spacy.io/) to preprocess the `text` column in the given dataframe `df` and save the processed text in a new column called `text_pp` within the same dataframe.

If you do not have spaCy in your course environment, you'll have to [install it](https://spacy.io/usage) and download the pretrained model en_core_web_md. 

`python -m spacy download en_core_web_md`


Note that there is no such thing as "perfect" preprocessing. You'll have to make your own judgments and decisions on which tokens are likely to be more informative for the given task. Some common text preprocessing steps for topic modeling include: 
- getting rid of slashes, new-line characters, or any other non-informative characters
- sentence segmentation and tokenization      
- replacing urls, email addresses, or numbers with generic tokens such as "URL",  "EMAIL", "NUM". 
- getting rid of other fairly unique tokens which are not going to help us in topic modeling  
- excluding stopwords and punctuation 
- lemmatization


> Check out [these available attributes](https://spacy.io/api/token#attributes) for `token` in spaCy which might help you with preprocessing. 

> You can also get rid of words with specific POS tags. [Here](https://universaldependencies.org/u/pos/) is the list of part-of-speech tags used in spaCy. 

> You may have to use regex to clean text before passing it to spaCy. Also, you might have to go back and forth between preprocessing in this exercise and and topic modeling in Exercise 2 before finalizing preprocessing steps. 

> Note that preprocessing the corpus might take some time. So here are a couple of suggestions: 1) During the debugging phase, work on a smaller subset of the data. 2) Once you finalize the preprocessing part, you might want to save the preprocessed data in a CSV and work with this CSV so that you don't run the preprocessing part every time you run the notebook. 
 


In [81]:
import spacy
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])

<div class="alert alert-warning">

Solution_2_1
    
</div>

_Points:_ 8

In [82]:
texts = df["text"].fillna("").astype(str).tolist()
clean_texts = []

for t in texts:
    t = t.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # Remove common newsgroup metadata noise
    t = re.sub(r"\b(from|subject|organization|lines|writes)\b\s*:?", " ", t, flags=re.IGNORECASE)

    # Remove url/email
    t = re.sub(r"https?://\S+|www\.\S+", " ", t)
    t = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", t)

    # Remove numbers
    t = re.sub(r"\b\d+([.,]\d+)?\b", " ", t)

    t = re.sub(r"[\\/|]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    clean_texts.append(t)

In [83]:
processed = []
keep_pos = {"NOUN", "PROPN", "ADJ"}  # Only keep nouns and adjectives and proper nouns

# Custom drop list based on manual inspection of the most common lemmas in the corpus
custom_drop = {
    "-pron-", "amp", "edu", "com", "don", "just", "like", "know", "think",
    "people", "say", "time", "good", "use", "make", "want", "post", "article",
    "write", "line", "organization", "subject"
}

# Process documents in batches
for doc in nlp.pipe(clean_texts, batch_size=256):
    tokens = []
    for tok in doc:
        if tok.is_stop or tok.is_punct or tok.is_space:
            continue
        if tok.pos_ not in keep_pos:
            continue
        if tok.like_url or tok.like_email or tok.like_num:
            continue

        lemma = tok.lemma_.lower().strip()

        if len(lemma) < 3:
            continue
        if lemma in custom_drop:
            continue
        if not re.fullmatch(r"[a-z][a-z\-]{2,24}", lemma):
            continue

        tokens.append(lemma)

    processed.append(" ".join(tokens))

In [84]:
df["text_pp"] = processed
df[["text_pp", "target_name"]].iloc[2:6]

,text_pp,target_name
2,actuallay hand idea newsgroup aspect graphic programming brian reply original posting loose structure reason group possible reason posting day posting interested aspect graphic meeting extension news forum idea difficult different thing meeting evening netter,comp.graphics
3,problem help window child buttonpressmask keypressmask child buttonpressmask keypressmask keypress event buttonpress event subwindow fyi xnew olvwm wrong,comp.windows.x
4,late upi foreign ministry spokesman ferhat ataman journalist turkey air space flight armenia humanitarian aid republic overland turkish territory uncivilized sign compassion humanitarian aid civilian population nazis turkey hypocrite condemnation serbians,talk.politics.mideast
5,leadership magazine disk paper disk illustration word processor christian magazine leadership disk medium info,soc.religion.christian


In [85]:
df.iloc[2:6]

,text,target,target_name,text_pp
2,"\nActuallay I don't, but on the other hand I don't support the idea of having\none newsgroup for every aspect of graphics programming as proposed by Brian,\nin his reply to my original posting.\nI would suggest a looser structure more like a comp.graphics.programmer,\ncomp.graphics.hw_and_sw\nThe reason for making as few groups as possible is for the same reason you\nsay we shouldn't spilt up, not to get to few postings every day.\nI takes to much time to browse through all postings just to find two or \nthree I'm interested in.\n\nI understand and agree when you say you want all aspects of graphics in one\nmeeting. I agree to some extension. I see news as a forum to exchange ideas,\nhelp others or to be helped. I think this is difficult to achive if there\nare so many different things in one meeting.\n\nGood evening netters|-)",1,comp.graphics,actuallay hand idea newsgroup aspect graphic programming brian reply original posting loose structure reason group possible reason posting day posting interested aspect graphic meeting extension news forum idea difficult different thing meeting evening netter
3,"The following problem is really bugging me,\nand I would appreciate any help.\n\nI create two windows:\n\nw1 (child to root) with event_mask = ButtonPressMask|KeyPressMask;\nw2 (child to w1) with do_not_propagate_mask = ButtonPressMask|KeyPressMask;\n\n\nKeypress events in w2 are discarded, but ButtonPress events fall through\nto w1, with subwindow set to w2.\n\nFYI, I'm using xnews/olvwm.\n\nAm I doing something fundamentally wrong here?",2,comp.windows.x,problem help window child buttonpressmask keypressmask child buttonpressmask keypressmask keypress event buttonpress event subwindow fyi xnew olvwm wrong
4,\n\n This is the latest from UPI \n\n Foreign Ministry spokesman Ferhat Ataman told journalists Turkey was\n closing its air space to all flights to and from Armenia and would\n prevent humanitarian aid from reaching the republic overland across\n Turkish territory.\n\n \n Historically even the most uncivilized of peoples have exhibited \n signs of compassion by allowing humanitarian aid to reach civilian\n populations. Even the Nazis did this much.\n\n It seems as though from now on Turkey will publicly pronounce \n themselves 'hypocrites' should they choose to continue their\n condemnation of the Serbians.\n\n\n\n--,7,talk.politics.mideast,late upi foreign ministry spokesman ferhat ataman journalist turkey air space flight armenia humanitarian aid republic overland turkish territory uncivilized sign compassion humanitarian aid civilian population nazis turkey hypocrite condemnation serbians
5,"Hi,\n I'd like to subscribe to Leadership Magazine but wonder if there is one on\ndisk instead of on paper. Having it on disk would save me retyping\nillustrations, etc into a word processor. It's just cut and paste.\n If there are other good Christian magazines like Leadership on disk media,\nI'd appreciate any info.",5,soc.religion.christian,leadership magazine disk paper disk illustration word processor christian magazine leadership disk medium info


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.2 Build a topic model using sklearn's LatentDirichletAllocation
rubric={points}

**Your tasks:**

1. Build LDA models on the preprocessed data using using [sklearn's `LatentDirichletAllocation`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html) and random state 42. Experiment with a few values for the number of topics (`n_components`). Pick a reasonable number for the number of topics and briefly justify your choice.

<div class="alert alert-warning">

Solution_2_2
    
</div>

_Points:_ 4

I chose 8 topics because the dataset has exactly 8 newsgroup categories, so 8 topics allows the model to find each category. The topics make sense and map clearly to the newsgroups (like hockey words for hockey, gun words for guns, etc.).

In [86]:
vectorizer = CountVectorizer(
    max_df=0.9,
    min_df=5,
    max_features=5000
)

X_pp = vectorizer.fit_transform(df["text_pp"])

topic_candidates = [6, 8, 7, 10, 12]
rows = []

# Fit LDA models for each candidate number of topics and compute log-likelihood and perplexity
for k in topic_candidates:
    lda_k = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        learning_method="batch",
        max_iter=20
    )
    lda_k.fit(X_pp)
    rows.append(
        {
            "n_topics": k,
            "train_log_likelihood": lda_k.score(X_pp),
            "train_perplexity": lda_k.perplexity(X_pp),
        }
    )

results_22 = pd.DataFrame(rows).sort_values("train_perplexity")
results_22

,n_topics,train_log_likelihood,train_perplexity
4,12,-1.797076e+06,1282.271503
3,10,-1.799163e+06,1292.972210
1,8,-1.809768e+06,1348.745588
2,7,-1.813182e+06,1367.206596
0,6,-1.817234e+06,1389.447068


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.3 Exploring word topic association
rubric={points}

**Your tasks:**
1. For the number of topics you picked in the previous exercise, show top 10 words for each of your topics and suggest labels for each of the topics (similar to how we came up with labels "health and nutrition", "fashion", and "machine learning" in the toy example we saw in class). 

> If your topics do not make much sense, you might have to go back to preprocessing in Exercise 2.1, improve it, and train your LDA model again. 

<div class="alert alert-warning">

Solution_2_3
    
</div>

_Points:_ 5

In [87]:
n_topics = 8

lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=20
)
lda_model.fit(X_pp)

feature_names = vectorizer.get_feature_names_out()
top_n = 10

topic_rows = []
for topic_idx, topic_weights in enumerate(lda_model.components_):
    top_word_idx = topic_weights.argsort()[::-1][:top_n]
    top_words = [feature_names[i] for i in top_word_idx]
    topic_rows.append(
        {
            "topic_id": topic_idx,
            "top_10_words": ", ".join(top_words)
        }
    )

topic_words_df = pd.DataFrame(topic_rows)
topic_words_df

,topic_id,top_10_words
0,0,"israel, jews, israeli, right, jewish, arab, war, state, human, country"
1,1,"god, jesus, thing, church, bible, way, christian, question, life, religion"
2,2,"child, woman, day, thing, way, fire, old, armenians, apartment, building"
3,3,"game, team, year, player, season, hockey, league, period, new, goal"
4,4,"file, program, window, entry, widget, color, jpeg, problem, error, application"
5,5,"gun, law, weapon, right, firearm, state, file, crime, government, year"
6,6,"armenian, turkish, armenians, turkey, turks, greek, genocide, armenia, government, history"
7,7,"image, available, server, system, version, software, file, program, graphic, motif"


In [88]:
topic_labels = {
    0: "Middle East (Israel/Jewish)",
    1: "Christianity",
    2: "Miscellaneous Events",  # Unclear mix: woman, child, fire, armenians
    3: "Hockey",
    4: "Graphics/Windows",
    5: "Gun Politics",
    6: "Middle East (Armenian/Turkish)",
    7: "Graphics/System/Software",
}

topic_words_df["suggested_label"] = topic_words_df["topic_id"].map(topic_labels)
topic_words_df

,topic_id,top_10_words,suggested_label
0,0,"israel, jews, israeli, right, jewish, arab, war, state, human, country",Middle East (Israel/Jewish)
1,1,"god, jesus, thing, church, bible, way, christian, question, life, religion",Christianity
2,2,"child, woman, day, thing, way, fire, old, armenians, apartment, building",Miscellaneous Events
3,3,"game, team, year, player, season, hockey, league, period, new, goal",Hockey
4,4,"file, program, window, entry, widget, color, jpeg, problem, error, application",Graphics/Windows
5,5,"gun, law, weapon, right, firearm, state, file, crime, government, year",Gun Politics
6,6,"armenian, turkish, armenians, turkey, turks, greek, genocide, armenia, government, history",Middle East (Armenian/Turkish)
7,7,"image, available, server, system, version, software, file, program, graphic, motif",Graphics/System/Software


<!-- END QUESTION -->

<br><br>

<!-- BEGIN QUESTION -->

### 2.4 Exploring document topic association
rubric={points}

**Your tasks:**
1. Show the document topic assignment of the first five documents from `df`.

<div class="alert alert-warning">

Solution_2_4
    
</div>

_Points:_ 5

In [89]:
# Document-topic distribution for all documents
doc_topic_dist = lda_model.transform(X_pp)

# Most likely topic per document
df["assigned_topic"] = doc_topic_dist.argmax(axis=1)

# Confidence of assignment
df["topic_confidence"] = doc_topic_dist.max(axis=1)

# Map readable labels
df["assigned_topic_label"] = df["assigned_topic"].map(topic_labels)

# Show first 5 document assignments
df.loc[:4, ["target_name", "assigned_topic", "assigned_topic_label", "topic_confidence", "text_pp"]]

,target_name,assigned_topic,assigned_topic_label,topic_confidence,text_pp
0,talk.politics.guns,5,Gun Politics,0.968729,paragraph unlawful person machinegun law dictionary person artificial entity government right federal constitution statute individual government law law law constitional court court run mill guilty jail right supreme court right
1,comp.graphics,4,Graphics/Windows,0.981347,bad question ref algorithm bit hard point plane circle algorithm center circle center perpendicular plane point center sphere unused point original point different sphere origin interection center sphere radius easy distance center original point math workable algorithm alternate method pair point plane perpendicular bisector segment pair center sphere pair plane point easy
2,comp.graphics,7,Graphics/System/Software,0.429785,actuallay hand idea newsgroup aspect graphic programming brian reply original posting loose structure reason group possible reason posting day posting interested aspect graphic meeting extension news forum idea difficult different thing meeting evening netter
3,comp.windows.x,4,Graphics/Windows,0.610160,problem help window child buttonpressmask keypressmask child buttonpressmask keypressmask keypress event buttonpress event subwindow fyi xnew olvwm wrong
4,talk.politics.mideast,6,Middle East (Armenian/Turkish),0.776408,late upi foreign ministry spokesman ferhat ataman journalist turkey air space flight armenia humanitarian aid republic overland turkish territory uncivilized sign compassion humanitarian aid civilian population nazis turkey hypocrite condemnation serbians


<!-- END QUESTION -->

<br><br><br><br>

<!-- BEGIN QUESTION -->

## Exercise 3: Short answer questions 
<hr>

rubric={points}

1. Briefly explain how content-based filtering works in the context of recommender systems. 
2. Discuss at least two negative consequences of recommender systems.
3. What is transfer learning in natural language processing? Briefly explain.     

<div class="alert alert-warning">

Solution_3
    
</div>

_Points:_ 6

1. Content-based filtering recommends items similar to what the user liked before by looking at item features. For example, if you watched an action movie, it recommends other action movies based on their shared characteristics.

2. First, they can create "filter bubbles" where users only see content similar to what they've already liked, limiting exposure to new ideas. Second, recommender systems can amplify biases in the training data and show unfair or stereotyped content to certain groups of people.

3. Transfer learning means using a model trained on a large dataset (like Wikipedia) to solve a new task with less data. For example, you can take a pre-trained word embedding model and use it to classify emails, instead of training from scratch on your small email dataset.

<!-- END QUESTION -->

<br><br><br><br>

Before submitting your assignment, please make sure you have followed all the instructions in the Submission Instructions section at the top. 

Here is a quick checklist before submitting: 

- [ ] Restart kernel, clear outputs, and run all cells from top to bottom.  
- [ ] `.ipynb` file runs without errors and contains all outputs.  
- [ ] Only `.ipynb` and required output files are uploaded (no extra files).  
- [ ] Execution numbers start at **1** and are in order.  
- [ ] If `.ipynb` is too large and doesn't render on Gradescope, also upload a PDF/HTML version.  
- [ ] Reviewed the [CPSC 330 homework instructions](https://ubc-cs.github.io/cpsc330-2025W2/docs/homework-instructions).  

![](img/eva-well-done.png)